In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision.datasets import FashionMNIST
from torchvision import transforms
from torch.utils.data import DataLoader

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

import numpy as n

In [ ]:
torch.manual_seed(123)
np.random.seed(123)

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = FashionMNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = FashionMNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [ ]:
class MLP(nn.Module):

    def __init__(self):
        super().__init__()

        self.model = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784,128),
            nn.ReLU(),
            nn.Linear(128,64),
            nn.ReLU(),
            nn.Linear(64,10)
        )

    def forward(self,x):
        return self.model(x)

In [ ]:
model = MLP()

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
epochs = 10

for epoch in range(epochs):

    model.train()

    for inputs, labels in train_loader:

        optimizer.zero_grad()

        outputs = model(inputs)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

In [ ]:
def evaluate_model(model, test_loader):

    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for inputs, labels in test_loader:

            outputs = model(inputs)

            _, preds = torch.max(outputs,1)

            all_preds.extend(preds.tolist())
            all_labels.extend(labels.tolist())

    return all_preds, all_labels


preds, labels = evaluate_model(model, test_loader)

In [ ]:
conf_matrix = confusion_matrix(labels, preds)

print("Confusion Matrix:")
print(conf_matrix)

In [ ]:
accuracy = accuracy_score(labels, preds)

precision = precision_score(labels, preds, average="macro")
recall = recall_score(labels, preds, average="macro")
f1 = f1_score(labels, preds, average="macro")

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)